# Convert Leica SCAPE Acquisitions

This notebook demonstrates how to convert raw Leica SCAPE acquisitions into a stitched OME-Zarr dataset using the `autoscape` workflow.

In [ ]:
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr.scene import NgffScene
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, CoordinateSystemIdentifier, Axis, Translation
from bioio import BioImage
import os
import numpy as np
import napari
import xmltodict
import tqdm
from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion
import glob
from pathlib import Path
import tqdm
#import ezomero
#from omero.model.enums import UnitsLength
#import omero

import zarr
zarr.config.set({"codec_pipeline.path": "zarrs.ZarrsCodecPipeline"})

In [2]:
viewer = napari.Viewer()

## Find all raw Leica SCAPE acquisitions

First, we need to locate all the raw Leica SCAPE acquisitions on our filesystem.
This can be done using the `glob` module to search for the appropriate file patterns.

In [8]:
root = r'E:\UserData\Johannes\20260512\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\TileScan 1'
file_names = glob.glob(os.path.join(root, "**", "*ome.tif"), recursive=True)
file_names

['E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\\TileScan 1\\E\\1.ome.tif',
 'E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\\TileScan 1\\E\\10.ome.tif',
 'E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\\TileScan 1\\E\\11.ome.tif',
 'E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\\TileScan 1\\E\\12.ome.tif',
 'E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\\TileScan 1\\E\\2.ome.tif',
 'E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\\TileScan 1\\E\\3.ome.tif',
 'E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\\TileScan 1\\E\\4.ome.tif',
 'E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Plate2_Experiment1001\\TileScan 1\\E\\5.ome.tif',
 'E:\\UserData\\Johannes\\20260512\\2026_05_12_10_26_00--20260507_Pla

In [9]:
def find_in_nested_dict(data, key, value):
    """
    Recursively search for objects containing a specific key-value pair.
    Returns a list of all matching objects.
    """
    results = []
    
    def search(obj):
        if isinstance(obj, dict):
            if obj.get(key) == value:
                results.append(obj)
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)
    
    search(data)
    return results  

## Convert to OME-Zarr

Once we have located all the raw acquisitions,
we can proceed to convert them into the OME-Zarr format using the `ome-zarr-py` library.

In [ ]:
write_jobs = []
images = []
transforms = []
for filename in tqdm.tqdm(file_names):
	img_basename = Path(filename).stem.split(".")[0] 
	outfile_oz = img_basename + ".ome.zarr"
	bf = BioImage(filename)
	

	if bf.ome_metadata.images[0].pixels.physical_size_z is None:
		size_z = 0
	else:
		size_z = bf.ome_metadata.images[0].pixels.physical_size_z

	scale = {
		"z": size_z,
		"y": bf.ome_metadata.images[0].pixels.physical_size_y,
		"x": bf.ome_metadata.images[0].pixels.physical_size_x
	}

	axes_units = {
		"z": "meter",
		"y": "meter",
		"x": "meter",
	}
	
	# parse tile metadata
	metadata_file = os.path.join(Path(filename).parent, "Metadata", f"{img_basename}.xlif")
	metadata_dict = xmltodict.parse(open(metadata_file).read())
	tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]

	# n_series= bf.series_count()
	n_series = len(bf.scenes)

	all_positions = []
	if n_series > 1:
		sims = []
		for idx in range(n_series):
			bf.set_scene(bf.scenes[idx])
			dask_image = bf.dask_data
			size_tile = np.asarray(dask_image.squeeze().shape) * np.asarray([scale["z"], scale["y"], scale["x"]])
			# dask_image = bf.to_dask(series=idx)
			ts_info = tile_scan_info[idx]

			t_z = float(ts_info["@PosZ"])
			t_y = float(ts_info["@PosY"])
			t_x = float(ts_info["@PosX"])

			all_positions.append(np.asarray([t_z, t_y, t_x]))

			sims.append(
				si_utils.get_sim_from_array(
					dask_image,
					dims=["t", "c", "z", "y", "x"],
					scale=scale,
					translation={"z": t_z, "y": t_y, "x": t_x},
					transform_key="stage_metadata",
					)
			)

		image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion)
	else:
		# image = bf.to_dask(series=0)
		image = bf.dask_data
		size_tile = np.asarray(dask_image.squeeze()) * np.asarray([scale["z"], scale["y"], scale["x"]])
		t_z = float(tile_scan_info["@PosZ"])
		t_y = float(tile_scan_info["@PosY"])
		t_x = float(tile_scan_info["@PosX"])

		all_positions.append(np.asarray([t_z, t_y, t_x]))

	all_positions = np.stack(all_positions, axis=0)
	position_corner = np.min(all_positions, axis=0) - size_tile / 2

	ngff_image = NgffImage(
		image.squeeze(),
		dims=["z", "y", "x"],
		scale=scale,
		axes_units=axes_units,
		name=Path(filename).parent.stem + img_basename
	)
	ngff_multiscales = NgffMultiscales(
		image=ngff_image,
	)
	images.append(ngff_multiscales)

	transform = Translation(
		translation=position_corner.tolist(),
		input=CoordinateSystemIdentifier(
			path=ngff_multiscales.name,
			name=ngff_multiscales.metadata.coordinateSystems[0].name
		),
		output=CoordinateSystemIdentifier(
			name="world"
		)
	)
	transforms.append(transform)

100%|██████████| 48/48 [00:54<00:00,  1.13s/it]


We need to define a common world coordinate system for all the acquisitions to ensure proper alignment and stitching.
World coordinates correspond to the microscope's motor coordinate system.
In the LEICA SCAPE system, the unit of measurement is typically meter.

In [ ]:
cs_world = CoordinateSystem(
	name="world",
	axes=(
		Axis(name="z", type="space", unit="meter"),
		Axis(name="y", type="space", unit="meter"),
		Axis(name="x", type="space", unit="meter"),
	)
)

ngff_scene = NgffScene(
	images=images,
	transformations=transforms,
	coordinate_systems=(
		cs_world,
	)
)

delayed_objects = ngff_scene.to_ome_zarr(
	os.path.join(root, "scene.ome.zarr")
)

## Show scenes



In [7]:
viewer.open(os.path.join(root, "scene.ome.zarr"), plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'A1'}, 'output': {'name': 'world'}, 'translation': [0.0028384652000001, 0.0057091052, 0.0074873634000000005]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'A10'}, 'output': {'name': 'world'}, 'translation': [0.0028384652000001, 0.0057091052, 0.0884873651]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'A11'}, 'output': {'name': 'world'}, 'translation': [0.0028384652000001, 0.0057091052, 0.0974873641]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'A12'}, 'output': {'name': 'world'}, 'translation': [0.0028384652000001, 0.0057091052, 0.10648736310000001]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'A2'}, 'output': {'name': 'world'}, 'translation': [0.0028384652000001, 0.0057091052, 0.0164873631]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'

[<Image layer 'scene.ome' at 0x27da4214470>,
 <Image layer 'scene.ome [1]' at 0x27da3c57500>,
 <Image layer 'scene.ome [2]' at 0x27d44f83e60>,
 <Image layer 'scene.ome [3]' at 0x27da43d2ba0>,
 <Image layer 'scene.ome [4]' at 0x27da988d700>]

## Optional: Upload tiles to OMERO

In [ ]:
host = 'omero-nfdi.uni-muenster.de'
port = 4064
user = 'JohannesSoltwedel'
group = None
secure=True

conn = ezomero.connect(host=host, user=user, secure=secure, port=port, group=group, password=password)

In [32]:
for layer in viewer.layers:
	projection = layer.data[0].min(axis=0).rechunk(2560, 2560)

	# iterate over chunks and save as individual images
	for idx in np.ndindex(projection.numblocks):
		chunk = projection.blocks[idx]
		if not chunk.shape == (2560, 2560):
			continue

		tile = chunk.compute()

		if sum(tile.flatten()==1) > 10000:
			continue

		image_id = ezomero.post_image(
			conn,
			image=tile[None, None, None],
			dataset_id=251,
			image_name=f"{layer.name}_chunk_{idx}", dim_order='tczyx', across_groups=False)
		
		conn.setChannelNames(data_type='Image', ids=[image_id], nameDict={1: 'Brightfield'})

		unit_x = omero.model.LengthI(scale["x"]* 10e6, UnitsLength.MICROMETER)
		unit_y = omero.model.LengthI(scale["y"]* 10e6, UnitsLength.MICROMETER)

		image_object = conn.getObject("Image", image_id)
		primary_pixels_object = image_object.getPrimaryPixels()._obj
		primary_pixels_object.setPhysicalSizeX(unit_x)
		primary_pixels_object.setPhysicalSizeY(unit_y)
		conn.getUpdateService().saveObject(primary_pixels_object)


In [29]:
1280 * 2

2560